# Multiclass GP分類とActive Learning：初学者向け実践ガイド

このNotebookでは、**3クラス分類**の学習から、確率予測、Active Learning、q-batch候補生成、Mixed入力まで順に試します。

- `posterior(X)` は**クラスごとの潜在GP**で、分類確率ではありません。
- `predict_proba(X)` は各クラスの事後予測確率です（最後の軸の合計が1）。
- `probability_posterior(X)` はSoftmax後のクラス確率の平均・事後分散・周辺信用区間を提供します。
- `PredictiveEntropy` は予測ラベルの不確実性、`BALD` はモデルについての情報獲得を重視します。
- `qPredictiveEntropy` は各候補のスコアを集約し、`qBALD` は候補間の依存を含む同時情報量を評価します。

関連：[分類GPの理論](../../docs/theory/30_multiclass_classification.md) / [分類GPガイド](../../docs/guide/30_multiclass_classification.md) / [獲得関数対応表](../../docs/models/multiclass-active-learning.md)

**回帰目的との組み合わせ：** 多クラス分類を制約として使い、単目的・多目的回帰の実験候補を提案する例は[61番：異種モデルによる分類制約付きBO](61_heterogeneous_classification_constraints.ipynb)にまとめています。


**独立した理論・実装ガイド：** [マルチクラスGP分類の理論（第30章）](../../docs/theory/30_multiclass_classification.md) / [マルチクラスGP分類ガイド](../../docs/guide/30_multiclass_classification.md)。二値分類の基礎は[第24章](../../docs/theory/24_classification.md)を参照してください。


## 最初に：このNotebookで何を解決するのか

たとえば製造設備の状態を「正常・注意・異常」の3種類に分けたいとします。
ラベル付けには人手や検査コストがかかるため、すべての条件を測るのではなく、
**次に調べると学習に役立つ条件**を選びたい、というのがActive Learningです。

ここで使う分類GPは、入力条件からクラスを直接断定するだけでなく、
「正常60%、注意30%、異常10%」のような**確率**を推定します。
予測が不確かな条件を優先して追加測定することで、少ないラベル数でも
分類器を改善できる可能性があります。ただし、必ず改善する保証はありません。

このNotebookは次の順で進みます。

1. 入力 `X` と正解ラベル `y` の形を確認する
2. GP分類器を学習し、3クラスの確率を読む
3. EntropyとBALDの違いを確かめる
4. 2点をまとめて提案する `q=2` を試す
5. カテゴリ変数を含むMixed入力を扱う

各セルの出力形状と意味を確認しながら進めてください。

### このNotebookで使う最適化API

分類Active Learningの候補提案では、クラス1の確率や事後平均を単純に最大化するのではなく、**分類用獲得関数**を最適化します。

- 連続入力の `q=2`：既存 `SobolSearchStrategy.optimize(qPredictiveEntropy(model), q=2)` を利用します。これはSobol候補の中で獲得関数を最大化する既存の最適化APIです。
- Mixed入力：カテゴリを固定した合法な候補集合を生成し、BoTorch公開 `optimize_acqf_discrete` で獲得関数を最大化します。カテゴリ0/1/2の中間値は探索しません。
- Ask-and-Tell：未ラベルの固定プールから選ぶ場合は `optimize_acqf_discrete` を利用し、選んだ候補をプールから除外します。

`qPredictiveEntropy` は各点の周辺Entropyを平均します。候補間の情報重複を明示的に差し引く `qBALD` とは目的が異なります。

## 1. 準備と3クラスの教師データ

ここでは2次元入力からクラス0・1・2を決める小さな人工データを使います。
乱数シードを固定し、再現可能な例にします。

In [ ]:
import torch

from robotorchan.acquisition import (
    BALD,
    PredictiveEntropy,
    qBALD,
    qPredictiveEntropy,
)
from robotorchan.models.classification import (
    MixedMulticlassSingleTaskGPClassifier,
    MulticlassSingleTaskGPClassifier,
)
from robotorchan.optim.strategies.sobol import SobolSearchStrategy

torch.manual_seed(17)
dtype = torch.double
train_X = torch.rand(24, 2, dtype=dtype)
train_Y = torch.bucketize(train_X[:, 0], torch.tensor([0.33, 0.66], dtype=dtype))
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=dtype)
print("入力:", train_X.shape, "ラベル:", train_Y.unique(sorted=True))

## 2. モデルの構築と学習

クラスごとの変分GPを利用します。ここではNotebookを短時間で実行できるよう学習回数を少なくしています。実務では検証データのNLL・Accuracy・校正などを確認し、学習条件を調整してください。

### 学習パラメータの意味

- `num_classes=3`：予測するクラスが3種類であることを示します。
- `inducing_points=8`：変分GPの近似に使う代表点数です。増やすと表現力が上がる可能性がありますが計算負荷も増えます。
- `epochs=5`：データを繰り返し学習する回数です。実務で5回が十分という意味ではありません。
- `lr=0.03`：最適化の学習率です。大きすぎると不安定になることがあります。

**重要：** この例はAPI確認用であり、予測精度の検証ではありません。

In [ ]:
model = MulticlassSingleTaskGPClassifier(
    train_X, train_Y, num_classes=3, inducing_points=8
)
model.fit(epochs=5, lr=0.03)
test_X = torch.tensor([[0.2, 0.4], [0.5, 0.4], [0.8, 0.4]], dtype=dtype)
probabilities = model.predict_proba(test_X, num_samples=32)
print("クラス確率:", probabilities)
print("予測クラス:", model.predict_class(test_X))
assert probabilities.shape == (3, 3)
assert torch.allclose(probabilities.sum(-1), torch.ones(3, dtype=dtype))

## 3. 確率と潜在値を混同しない

`posterior` は3クラスに対応する潜在関数の分布です。softmax後の確率の事後分布は `probability_posterior` で取得します。

In [ ]:
latent = model.posterior(test_X)
print("潜在GP平均:", latent.mean.shape)
print("クラス確率:", probabilities.shape)
print("予測エントロピー:", model.predictive_entropy(test_X, num_samples=32))

### 3.1 確率Posteriorの平均・分散・周辺信用区間

`model.posterior(X)` が返すのは**潜在GP**で、クラス確率ではありません。
`model.probability_posterior(X)` の `mean` / `variance` は
Softmax変換した**確率ベクトルの事後モーメント**です。
`num_samples` と `seed` を指定すると、BoTorchのSobol QMCで平均と分散を
**同じ潜在サンプル**から計算します。これは厳密な解析解ではありません。

`credible_interval(0.90, num_samples=512)` は、クラスごとに確率サンプルの
5%点と95%点を取った**周辺90%信用区間**です。
各クラスの区間を同時に満たす確率が90%という意味ではありません。
`predict_proba` は別のMonte Carlo積分なので、有限標本では
`probability_posterior.mean` と完全一致するとは限りません。


In [ ]:
import matplotlib.pyplot as plt

# 第2特徴量を0.4に固定し、第1特徴量を動かして各クラスの確率を描きます。
slice_x = torch.linspace(0.05, 0.95, 41, dtype=dtype)
slice_X = torch.stack((slice_x, torch.full_like(slice_x, 0.4)), dim=-1)
with torch.no_grad():
    probability_posterior = model.probability_posterior(
        slice_X, num_samples=256, seed=17
    )
    class_mean = probability_posterior.mean
    class_variance = probability_posterior.variance
    class_lower, class_upper = probability_posterior.credible_interval(
        0.90, num_samples=512
    )
    probability_samples = probability_posterior.rsample(torch.Size([8]))

assert class_mean.shape == class_variance.shape == (41, 3)
assert class_lower.shape == class_upper.shape == (41, 3)
torch.testing.assert_close(class_mean.sum(-1), torch.ones(41, dtype=dtype))
torch.testing.assert_close(
    probability_samples.sum(-1), torch.ones_like(probability_samples[..., 0])
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for cls in range(model.num_classes):
    x = slice_x.numpy()
    axes[0].plot(x, class_mean[:, cls].numpy(), label=f"Class {cls} mean")
    axes[0].fill_between(
        x, class_lower[:, cls].numpy(), class_upper[:, cls].numpy(), alpha=0.15
    )
    axes[1].plot(
        x, class_variance[:, cls].numpy(), label=f"Class {cls} variance"
    )
axes[0].set_ylabel("Class probability")
axes[1].set_ylabel("Epistemic probability variance")
axes[0].set_ylim(-0.05, 1.05)
for ax in axes:
    ax.set_xlabel("First feature (second = 0.4)")
    ax.legend()
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


### 予測エントロピーを数式で理解する

クラス確率を \(p_0,p_1,p_2\) とすると、

\[
H(y\mid x,D)=-\sum_{c=0}^{2}p_c\log p_c
\]

です。3クラスがほぼ同確率ならエントロピーは大きく、
1クラスの確率がほぼ1なら小さくなります。

一方、BALDは「ラベルの曖昧さのうち、**モデルがまだ学習できていない部分**」
を重視します。データ自体が曖昧な点と、追加観測で学習が進みそうな点は
必ずしも一致しません。

単一候補用の獲得関数には通常 \`X.shape == [候補数, 1, 入力次元]\`
を渡します。真ん中の \`1\` が \`q=1\` に相当します。

## 4. 単一候補のActive Learning

未観測点のうち、どの点にラベルを付けるとよいかをスコアで比較します。
ここではエントロピーとBALDを使います。両者の順位が異なることがあります。

In [ ]:
pool = torch.rand(16, 2, dtype=dtype)
entropy = PredictiveEntropy(model)
bald = BALD(model, num_samples=16)
entropy_scores = entropy(pool.unsqueeze(-2))
bald_scores = bald(pool.unsqueeze(-2))
print("Entropy候補:", pool[entropy_scores.argmax()])
print("BALD候補:", pool[bald_scores.argmax()])
assert entropy_scores.shape == (16,)
assert bald_scores.shape == (16,)

### 既存の `optimize()` を使う利点

`SobolSearchStrategy.optimize()` は `qPredictiveEntropy` などの獲得関数を受け取り、探索範囲内のSobol候補から最も高い獲得値を持つバッチを返します。`result.candidates` は候補の入力値、`result.acquisition_value` は獲得関数値です。

この方法は勾配に依存しないため、モンテカルロ推定を含む分類獲得関数でも扱いやすい一方、探索点数によって精度・計算時間が変わります。

### q-batchの考え方

`q=2` は、**次に測定する条件を2つまとめて選ぶ**ことです。
`qPredictiveEntropy` は各候補のスコアを集約するため、
候補同士の重複情報を直接は評価しません。
`qBALD` は候補の同時ラベル分布を使い、候補間の依存を考慮します。

クラス数を `C` とすると、qBALDが調べるラベルの組合せは `C**q` 個です。
3クラスで `q=2` なら9通り、`q=5` なら243通りです。
実装は256通りまでに制限しているため、qを大きくすればよいわけではありません。

`SobolSearchStrategy` は獲得関数のスコアが高い候補を探します。
返された候補は**推奨実験条件**であり、正解ラベルの予測値ではありません。

## 5. q=2の候補をまとめて提案

`SobolSearchStrategy` は既存の獲得関数を受け取り、探索範囲内の候補を返します。
`qBALD` はラベルの組合せを列挙するため、クラス数をC、候補数をqとするとC**qが256以下である必要があります。

In [ ]:
strategy = SobolSearchStrategy(bounds, num_samples=16, seed=17)
result = strategy.optimize(qPredictiveEntropy(model), q=2)
print("qPredictiveEntropy候補:", result.candidates)
print("獲得関数値:", result.acquisition_value)
joint_scores = qBALD(model, num_samples=16)(result.candidates.unsqueeze(0))
print("同じ候補のjoint qBALD:", joint_scores)
assert result.candidates.shape == (2, 2)
assert joint_scores.shape == (1,)

## 6. Mixed入力（連続値＋カテゴリ）

第2列をカテゴリ0・1・2に変更し、`cat_dims=[1]` を指定します。
モデル側のカテゴリ設定と、候補生成側の離散変数指定は別々の責務です。
ここでは合法なカテゴリ候補のスコア評価を確認します。

In [ ]:
mixed_X = train_X.clone()
mixed_X[:, 1] = torch.arange(len(mixed_X)) % 3
mixed_Y = (torch.arange(len(mixed_X)) % 3).long()
mixed_model = MixedMulticlassSingleTaskGPClassifier(
    mixed_X, mixed_Y, num_classes=3, cat_dims=[1], inducing_points=8
)
mixed_model.fit(epochs=5, lr=0.03)
mixed_candidates = torch.tensor(
    [[[0.2, 0.0], [0.7, 1.0]], [[0.4, 2.0], [0.9, 0.0]]], dtype=dtype
)
mixed_scores = qPredictiveEntropy(mixed_model)(mixed_candidates)
print("Mixed q-batch scores:", mixed_scores)
assert mixed_scores.shape == (2,)

### Mixed入力でも候補を**最適化**する

前のセルはMixed候補のスコアを計算するだけでした。ここではBoTorchの公開 `optimize_acqf_discrete` を使い、**カテゴリ0・1・2を満たす候補集合からEntropyの高い点を実際に提案**します。

カテゴリは列挙し、連続変数だけを格子状に動かします。したがってカテゴリに対して微分を行いません。これは有限候補集合の最適化であり、連続空間全体の厳密最適化ではありません。

In [ ]:
from botorch.optim import optimize_acqf_discrete

mixed_x_values = torch.linspace(0.0, 1.0, 31, dtype=dtype)
mixed_choice_pool = torch.cartesian_prod(
    mixed_x_values, torch.tensor([0.0, 1.0, 2.0], dtype=dtype)
)
mixed_entropy_acq = PredictiveEntropy(mixed_model)
mixed_next_X, mixed_next_value = optimize_acqf_discrete(
    acq_function=mixed_entropy_acq,
    q=1,
    choices=mixed_choice_pool,
    unique=True,
)
assert mixed_next_X.shape == (1, 2)
assert mixed_next_X[0, 1].item() in (0.0, 1.0, 2.0)
print("Mixed optimal candidate:", mixed_next_X.detach())
print("PredictiveEntropy score:", mixed_next_value.detach())

# 合成データではカテゴリIDを正解ラベルとする例。実運用では測定・人手判定で取得する。
mixed_observed_y = mixed_next_X[:, 1].to(dtype=torch.long)
mixed_updated_X = torch.cat((mixed_X, mixed_next_X.detach()), dim=0)
mixed_updated_y = torch.cat((mixed_Y, mixed_observed_y), dim=0)
mixed_updated_model = MixedMulticlassSingleTaskGPClassifier(
    mixed_updated_X, mixed_updated_y,
    num_classes=3, cat_dims=[1], inducing_points=8,
)
mixed_updated_model.fit(epochs=5, lr=0.03)
print("Mixed observations before / after:", len(mixed_Y), len(mixed_updated_y))

### Mixed入力で注意する点

連続変数は数値の近さを使って比較できますが、
カテゴリ変数の値0・1・2は必ずしも「距離1」の意味を持ちません。
`cat_dims=[1]` は、2列目をカテゴリとして扱う指定です。

分類器に `cat_dims` を設定することと、
候補生成器に合法なカテゴリ値を教えることは**別の操作**です。
今回のセルは合法なカテゴリ値を持つ候補のスコア評価を示しています。
Mixed探索全体の設定例は
[獲得関数の対応表](../../docs/models/multiclass-active-learning.md)を参照してください。

## 7. 実際に次の実験を提案する：Ask-and-Tellを2回繰り返す

ここまでの `entropy_scores` は**候補の評価値**でした。実験計画として使うには、
(1) 未測定候補から選ぶ → (2) 実験・人手でラベルを得る → (3) 学習データに追加する
→ (4) 再学習して再提案、という一連の流れが必要です。

ここでは実験の代わりに、最初に定義した人工的な判定規則
（第1特徴量が0.33未満なら0、0.66未満なら1、それ以外は2）を
**ラベル取得装置の代役**にします。実務ではこの行を検査結果・専門家の判定に置き換えます。
ラベルをまだ知らない候補プールから選ぶため、未測定データの正解ラベルは
**選択が終わるまで参照しません**。

学習のたびにモデルを新しく構築し、すでに測定した候補はプールから削除します。
今回は説明用に `q=1` とします。同時に2点測る場合は第5節の `q=2` を使います。
獲得関数の値は「正解確率」ではなく、次にラベルを取得する価値の指標です。

### 未ラベル候補プールの最適化にもBoTorchを使う

`optimize_acqf_discrete(q=1, choices=unlabeled_pool)` は、与えられた**合法な未ラベル候補の集合**から獲得関数を最大化します。選択後にその行をプールから削除すれば、同じ候補を繰り返し問い合わせません。

この場合は勾配による連続最適化よりも、候補集合を明示する離散最適化が自然です。分類獲得関数は既存の `PredictiveEntropy` をそのまま使います。

In [ ]:
# 未測定候補を固定し、各ラウンドで未測定の候補だけから提案します。
unlabeled_pool = torch.rand(24, 2, dtype=dtype)
observed_X = train_X.clone()
observed_y = train_Y.clone()
for round_index in range(2):
    current_model = MulticlassSingleTaskGPClassifier(
        observed_X, observed_y, num_classes=3, inducing_points=8
    )
    current_model.fit(epochs=5, lr=0.03)

    # PredictiveEntropy: クラス確率の曖昧さが大きい候補を優先します。
    acquisition = PredictiveEntropy(current_model)
    next_X, selected_value = optimize_acqf_discrete(
        acq_function=acquisition,
        q=1,
        choices=unlabeled_pool,
        unique=True,
    )
    # プールから返された行を特定し、次のラウンドでは除外する
    chosen_index = torch.where(torch.all(unlabeled_pool == next_X[0], dim=-1))[0][0].item()
    print(f"Round {round_index + 1}: 提案条件 = {next_X.tolist()}")
    print(f"  提案スコア = {selected_value.detach().item():.4f}")

    # ここが実験・検査・人手のラベル付けに相当します。
    # 実際の未知ラベルは提案後にのみ取得してください。
    next_y = torch.bucketize(
        next_X[:, 0], torch.tensor([0.33, 0.66], dtype=dtype)
    )
    print(f"  取得したラベル = {next_y.tolist()}")

    observed_X = torch.cat([observed_X, next_X], dim=0)
    observed_y = torch.cat([observed_y, next_y], dim=0)
    unlabeled_pool = torch.cat(
        [unlabeled_pool[:chosen_index], unlabeled_pool[chosen_index + 1 :]], dim=0
    )
    print(f"  ラベル付き件数 = {len(observed_y)} / 未測定候補 = {len(unlabeled_pool)}")

assert len(observed_y) == len(train_Y) + 2
assert len(unlabeled_pool) == 22


## 8. 次の実験につなげる

提案された候補に対して実際にラベルを観測したら、学習データへ追加し、
モデルを再学習して候補提案を繰り返します。

**注意：** このNotebookはAPIと計算経路を理解するための例です。
性能の優劣を示すベンチマークではありません。比較には複数シード、
ラベル予算、ホールドアウトデータ、Accuracy・NLL・Brier scoreなどが必要です。

## 9. 実務での確認項目

- **分割方法**：評価データを学習データから分離する。時系列データなら未来情報の混入を避ける。
- **クラス不均衡**：少数クラスのRecall、混同行列、クラス別の予測確率を確認する。
- **確率校正**：AccuracyだけでなくNLLやBrier scoreも見る。
- **ラベル予算**：同じ予算・初期データ・乱数シードでRandom Samplingと比較する。
- **再現性**：複数シードの平均とばらつきを記録する。
- **制約**：実験不可能な条件は候補生成の入力制約で除外する。

**次に読む資料：**
[分類GPの理論](../../docs/theory/24_classification.md)、
[分類モデルのGuide](../../docs/guide/24_classification.md)、
[Active Learningの対応表](../../docs/models/multiclass-active-learning.md)。

## 10. 保留データでAccuracy・NLL・Brier Scoreを評価する

予測クラスが合っている割合（Accuracy）だけでは、**各クラスへ割り当てた予測確率の妥当性**は確認できません。学習・候補選択に使用していない人工的な保留データを作り、NLLと多クラスBrier Scoreも計算します。

- **NLL**：正解クラスの予測確率が低いほど大きくなり、過度に自信を持った誤分類を強く罰します。
- **Brier Score**：全クラスの予測確率と正解のone-hotベクトルとの差の二乗和を平均します。本例はクラス数で割らない定義です。
- **Accuracy**：確率最大のクラスが正解と一致した割合です。

3クラスの確率が `(0.7, 0.2, 0.1)` で正解がクラス0なら、1件分のBrier Scoreは `(0.7-1)^2 + 0.2^2 + 0.1^2 = 0.14` です。

予測に用いるのは第2節で学習した `model` です。**この保留データはモデルの学習やActive Learningの候補選択に使っていません。** 学習回数5回の例であり、良い数値や校正性能を保証するものではありません。

In [ ]:
# 学習・Active Learningの候補選択に使っていない別の入力を用意します。
holdout_x0 = torch.linspace(0.04, 0.96, 18, dtype=dtype)
holdout_X = torch.stack(
    (holdout_x0, torch.full_like(holdout_x0, 0.3)), dim=-1
)
holdout_Y = torch.bucketize(
    holdout_x0, torch.tensor([0.33, 0.66], dtype=dtype)
)

with torch.no_grad():
    holdout_proba = model.predict_proba(holdout_X, num_samples=256)
    holdout_pred = holdout_proba.argmax(dim=-1)
    accuracy = (holdout_pred == holdout_Y).to(dtype).mean()
    correct_proba = holdout_proba.gather(-1, holdout_Y.unsqueeze(-1)).squeeze(-1)
    nll = -correct_proba.clamp_min(1e-12).log().mean()
    one_hot = torch.nn.functional.one_hot(holdout_Y, num_classes=3).to(dtype)
    brier = ((holdout_proba - one_hot) ** 2).sum(dim=-1).mean()

assert holdout_proba.shape == (18, 3)
assert torch.isfinite(torch.stack((accuracy, nll, brier))).all()
print(f"Accuracy: {accuracy.item():.4f}")
print(f"NLL:      {nll.item():.4f}")
print(f"Brier:    {brier.item():.4f}（多クラスの二乗和、クラス数では割らない）")

## 11. まとめ：Multiclass Active Learningと最適化API

| 処理 | API |
|---|---|
| 3クラス分類 | `MulticlassSingleTaskGPClassifier` |
| Mixed 3クラス分類 | `MixedMulticlassSingleTaskGPClassifier` |
| 学習・再学習 | `model.fit()` |
| クラス確率 | `predict_proba()` / `probability_posterior()` |
| 分類獲得関数 | `PredictiveEntropy` / `BALD` / `qPredictiveEntropy` / `qBALD` |
| 連続q-batch最適化 | `SobolSearchStrategy.optimize()` |
| Mixed合法候補最適化 | `optimize_acqf_discrete()` |
| 固定未ラベルプールからの提案 | `optimize_acqf_discrete()` |
| 更新 | ラベル取得 → データ追加 → `fit()` |

**このNotebookは、スコアを計算するだけでなく、既存の最適化APIを使って次の実験条件を提案します。**

## Phase 4：非定常・Spectral Mixtureの比較
GibbsカーネルとSpectral Mixtureカーネルを通常入力・Mixed入力で比較します。各モデルを学習し、クラス確率を可視化して予測エントロピー最大の次候補を提案します。少数epochはAPIの動作例であり、予測性能の保証ではありません。


In [ ]:
# Phase 4: Gibbs nonstationary / Spectral Mixture — fit → probability → candidate
import torch
import matplotlib.pyplot as plt
from robotorchan.models.classification import (
    NonstationaryMulticlassGPClassifier,
    SpectralMixtureMulticlassGPClassifier,
    MixedNonstationaryMulticlassGPClassifier,
    MixedSpectralMixtureMulticlassGPClassifier,
)

torch.manual_seed(12)
x_cont = torch.linspace(-1, 1, 18, dtype=torch.double).unsqueeze(-1)
y_phase4 = torch.bucketize(
    x_cont.squeeze(-1), torch.tensor([-0.3, 0.3], dtype=torch.double)
)
grid = torch.linspace(-1, 1, 61, dtype=torch.double).unsqueeze(-1)
models_phase4 = (
    (NonstationaryMulticlassGPClassifier, False),
    (SpectralMixtureMulticlassGPClassifier, False),
    (MixedNonstationaryMulticlassGPClassifier, True),
    (MixedSpectralMixtureMulticlassGPClassifier, True),
)
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True, sharey=True)
for ax, (model_type, mixed) in zip(axes.flat, models_phase4):
    # Mixed例はカテゴリ0を固定した条件付き断面を可視化
    x = torch.cat((x_cont, (torch.arange(18) % 2).to(x_cont).unsqueeze(-1)), dim=-1) if mixed else x_cont
    candidate_x = torch.cat((grid, torch.zeros_like(grid)), dim=-1) if mixed else grid
    kwargs = {"cat_dims": [1]} if mixed else {}
    torch.manual_seed(42)  # モデルごとの乱数状態を固定し比較を再現可能にする
    if "SpectralMixture" in model_type.__name__:
        kwargs["num_mixtures"] = 2
    model = model_type(x, y_phase4, num_classes=3, **kwargs)
    model.fit(epochs=2, lr=0.001)
    with torch.no_grad():
        p = model.predict_proba(candidate_x)
        entropy = -torch.special.xlogy(p, p).sum(dim=-1)
        best_index = entropy.argmax()
        proposed = candidate_x[best_index]
    for c in range(3):
        ax.plot(grid.squeeze(-1).numpy(), p[:, c].numpy(), label=f"class {c}")
    ax.axvline(float(proposed[0]), color="black", linestyle="--", label="next candidate")
    ax.set_title(model_type.__name__)
    ax.set_ylabel("predicted probability")
    print(model_type.__name__, "candidate:", proposed.tolist())
axes[0, 0].legend(loc="best")
fig.tight_layout()
plt.show()


## Phase 5：高次元GP分類（PCA・PLS・Random Projection・ALEBO）

この節では **8次元の元入力 → 3次元の縮約入力 → fit → クラス確率 → 候補提案** を比較します。PCA・PLS・Random Projectionでは元の8次元候補をモデルへ渡します。ALEBOでは**埋め込み座標**を直接モデルへ渡します（実験空間への逆写像は別途 `ALEBOStrategy` が担当）。

ここでは学習が短時間で終わるようにepochを抑えます。PLSはラベルの数値化に依存し、特に名義尺度の多クラスではクラス番号の置換で縮約結果が変わる可能性があります。性能比較には独立した評価データが必要です。


In [ ]:
import torch
import matplotlib.pyplot as plt
from robotorchan.models.classification import (
    ALEBOMulticlassGPClassifier,
    PCAMulticlassGPClassifier,
    PLSMulticlassGPClassifier,
    RandomProjectionMulticlassGPClassifier,
)

torch.manual_seed(25)
train_X_hd = torch.rand(24, 8, dtype=torch.double)
# 3クラスの模擬観測。実際の利用では測定済みラベルに置き換えます。
train_Y_hd = torch.bucketize(train_X_hd[:, 0] + 0.5 * train_X_hd[:, 1],
                             torch.tensor([0.55, 0.95], dtype=torch.double))
candidate_X_hd = torch.rand(64, 8, dtype=torch.double)

models_hd = {
    "PCA": PCAMulticlassGPClassifier(train_X_hd, train_Y_hd, num_classes=3, n_components=3),
    "PLS": PLSMulticlassGPClassifier(train_X_hd, train_Y_hd, num_classes=3, n_components=3),
    "Random Projection": RandomProjectionMulticlassGPClassifier(
        train_X_hd, train_Y_hd, num_classes=3, n_components=3
    ),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, (name, model_hd) in zip(axes, models_hd.items()):
    model_hd.fit(epochs=2, lr=0.01)
    with torch.no_grad():
        p_hd = model_hd.predict_proba(candidate_X_hd)
        h_hd = -torch.special.xlogy(p_hd, p_hd).sum(-1)
        chosen_hd = int(h_hd.argmax())
    assert p_hd.shape == (64, 3)
    assert torch.allclose(p_hd.sum(-1), torch.ones(64, dtype=p_hd.dtype), atol=1e-5)
    ax.scatter(candidate_X_hd[:, 0].numpy(), h_hd.numpy(), s=14)
    ax.scatter(candidate_X_hd[chosen_hd, 0].item(), h_hd[chosen_hd].item(),
               marker="*", s=120, label="proposed")
    ax.set(title=name, xlabel="original input x[0]", ylabel="predictive entropy")
    ax.legend()
    print(name, "proposed original-space X:", candidate_X_hd[chosen_hd].tolist(),
          "class probabilities:", p_hd[chosen_hd].tolist())
plt.tight_layout()
plt.show()

# ALEBOモデルは元の8次元ではなく、埋め込みの3次元座標を受け取ります。
train_Z_hd = train_X_hd[:, :3].clone()
candidate_Z_hd = candidate_X_hd[:, :3].clone()
alebo_hd = ALEBOMulticlassGPClassifier(
    train_Z_hd, train_Y_hd, num_classes=3, projection=torch.eye(3, dtype=torch.double)
)
alebo_hd.fit(epochs=2, lr=0.01)
with torch.no_grad():
    alebo_p_hd = alebo_hd.predict_proba(candidate_Z_hd)
    alebo_h_hd = -torch.special.xlogy(alebo_p_hd, alebo_p_hd).sum(-1)
    alebo_idx_hd = int(alebo_h_hd.argmax())
assert alebo_p_hd.shape == (64, 3)
print("ALEBO proposed embedded Z:", candidate_Z_hd[alebo_idx_hd].tolist())
print("ALEBO class probabilities:", alebo_p_hd[alebo_idx_hd].tolist())
plt.figure(figsize=(6, 3))
plt.scatter(candidate_Z_hd[:, 0].numpy(), alebo_h_hd.numpy(), s=14)
plt.scatter(candidate_Z_hd[alebo_idx_hd, 0].item(), alebo_h_hd[alebo_idx_hd].item(),
            marker="*", s=120)
plt.xlabel("embedded coordinate z[0]")
plt.ylabel("predictive entropy")
plt.title("ALEBO embedded-space candidate")
plt.tight_layout()
plt.show()


## Phase 6：SAAS型・MAP-SAAS型の高次元GP分類

ここでは**元の8次元入力**で疎ARD事前分布を利用するモデルを比較します。
縮約（PCA等）とは異なり、入力の各次元にARD長さ尺度があり、逆長さ尺度に縮小事前分布を置きます。
両モデルとも既存の変分分類用 `fit()` を利用します。**完全Bayesian MCMCではありません。**

以下では学習、クラス確率、予測エントロピー、プール中の最大エントロピー候補提案、
散布図による確認まで実行します。学習epoch数はAPI動作確認用であり、性能を評価する値ではありません。


In [ ]:
import matplotlib.pyplot as plt
import torch

from robotorchan.models.classification import (
    MapSaasMulticlassGPClassifier,
    SaasMulticlassGPClassifier,
)

torch.manual_seed(36)
train_X_sparse = torch.rand(18, 8, dtype=torch.double)
train_Y_sparse = torch.bucketize(
    train_X_sparse[:, 0] + 0.3 * train_X_sparse[:, 1],
    torch.tensor([0.55, 0.9], dtype=torch.double),
)
candidate_X_sparse = torch.rand(64, 8, dtype=torch.double)

sparse_models = {
    "SAAS-style": SaasMulticlassGPClassifier(
        train_X_sparse, train_Y_sparse, num_classes=3, global_scale=0.1
    ),
    "MAP-SAAS-style": MapSaasMulticlassGPClassifier(
        train_X_sparse, train_Y_sparse, num_classes=3, tau=0.1
    ),
}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, sparse_model) in zip(axes, sparse_models.items()):
    sparse_model.fit(epochs=2, lr=0.005)
    with torch.no_grad():
        probabilities_sparse = sparse_model.predict_proba(candidate_X_sparse)
        entropy_sparse = -torch.special.xlogy(
            probabilities_sparse, probabilities_sparse
        ).sum(dim=-1)
        choice_sparse = int(entropy_sparse.argmax())

    assert probabilities_sparse.shape == (64, 3)
    assert torch.allclose(
        probabilities_sparse.sum(dim=-1),
        torch.ones(64, dtype=torch.double),
        atol=1e-5,
    )
    ax.scatter(candidate_X_sparse[:, 0].numpy(), entropy_sparse.numpy(), s=16)
    ax.scatter(
        candidate_X_sparse[choice_sparse, 0].item(),
        entropy_sparse[choice_sparse].item(),
        marker="*",
        s=120,
        label="proposed",
    )
    ax.set(title=name, xlabel="feature x[0]", ylabel="predictive entropy")
    ax.legend()
    print(name, "candidate:", candidate_X_sparse[choice_sparse].tolist())
    print(name, "class probability:", probabilities_sparse[choice_sparse].tolist())
plt.tight_layout()
plt.show()


### Phase 6：学習後のARD長さ尺度を確認する

SAAS型事前分布は各入力次元の**逆長さ尺度**（1 / lengthscale）を小さくする方向に働きます。値が大きい次元は、モデルの出力がその方向に敏感になり得ます。ただし逆長さ尺度は因果的な重要度ではなく、特徴量のスケールや相関にも左右されます。比較前に入力範囲を揃え、保留データの予測性能と併せて解釈します。


In [ ]:
from gpytorch.kernels import ScaleKernel

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for ax, (name, sparse_model) in zip(axes, sparse_models.items()):
    for class_index, class_gp in enumerate(sparse_model.class_models):
        covar = class_gp.model.covar_module
        base = covar.base_kernel if isinstance(covar, ScaleKernel) else covar
        inv_lengthscale = base.lengthscale.detach().cpu().reciprocal().reshape(-1)
        ax.plot(
            range(len(inv_lengthscale)),
            inv_lengthscale.numpy(),
            marker="o",
            label=f"class {class_index}",
        )
    ax.set(title=name, xlabel="input feature index", ylabel="inverse lengthscale")
    ax.legend()
plt.tight_layout()
plt.show()
